In [17]:
import requests, csv
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from urllib.request import Request, urlopen
import re
import subprocess
from tqdm import tqdm
import json
from difflib import SequenceMatcher
from urllib.parse import quote
import undetected_chromedriver as uc
import time
import random
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

In [6]:
HEADER_STYLES = {
      'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.84 Safari/537.36',
      'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
      'Accept-Charset': 'ISO-8859-1,latin-1;q=0.7,*;q=0.3',
      'Accept-Encoding': 'none',
      'Accept-Language': 'fr;q=0.9,en-US,en;q=0.8',
      'Connection': 'keep-alive',
      'referer': 'https://imdb.com'
}

In [3]:
def get_chrome_version_windows():
    try:
        # Commande pour interroger le registre Windows
        cmd = r'reg query "HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon" /v version'
        output = subprocess.check_output(cmd, shell=True).decode()
        version = re.search(r'REG_SZ\s+([\d.]+)', output)
        return version.group(1) if version else "Version non trouvée"
    except Exception as e:
        return f"Erreur : {e}"



def init_driver(chrome_version=None):
    """
    Initialise et retourne un navigateur undetected_chromedriver configuré
    avec les en-têtes HTTP personnalisés (ex: langue française).
    """

    options = uc.ChromeOptions()

    # Lancement du driver
    driver = uc.Chrome(options=options, version_main=int(chrome_version.split(".")[0]), use_subprocess=True)
    # Activation du protocole CDP (Chrome DevTools Protocol)
    driver.execute_cdp_cmd("Network.enable", {})

    # Configuration des headers personnalisés
    driver.execute_cdp_cmd("Network.setExtraHTTPHeaders", {
        "headers": {
            "Accept-Language": "en-US,en;q=0.9"
        }
    })

    return driver

In [4]:
def date_convertor(date_str):
    # Dictionnaire des mois en français
    month_fr = {
        'janvier': 1, 'février': 2, 'mars': 3, 'avril': 4,
        'mai': 5, 'juin': 6, 'juillet': 7, 'août': 8,
        'septembre': 9, 'octobre': 10, 'novembre': 11, 'décembre': 12
    }
    
    # Nettoyage de la chaîne (minuscules, suppression des espaces superflus)
    parts = date_str.strip().lower().split()
    
    day = int(parts[0])
    month = month_fr.get(parts[1])
    year = int(parts[2])
        
    dt = datetime(year, month, day)
    return dt.strftime('%Y-%m-%d')


def get_month_movies(year, month):

    month_dict = {'january':'01', 'february':'02', 'march':'03', 'april':'04', 'may':'05', 'june':'06', \
                'july':'07', 'august':'08', 'september':'09', 'october':'10', 'november':'11', 'december':'12'}

    req1 = Request(f"https://www.allocine.fr/film/agenda/mois/mois-{year}-{month_dict[month]}", headers=HEADER_STYLES)
    webpage1 = urlopen(req1).read()
    bs = BeautifulSoup(webpage1, "html.parser")

    results = []
    blocks = bs.find_all('div', class_='movie-agenda-month')

    for block in blocks:
        # 1. On récupère le titre (h2) de la semaine
        h2 = block.find('h2', class_='title-inter')
        if h2:
            week = date_convertor(h2.get_text(strip=True).replace("Films sortis en salles la semaine du ",""))
        
        # 2. On récupère tous les films (li) à l'intérieur de ce bloc
        li_films = block.find_all('li', class_='month-movie-item')
        
        for li in li_films:
            a_tag = li.find('a', class_='month-movies-link')
            if a_tag:
                title = a_tag.get_text(strip=True)
                spans = li.find_all('span', class_='light')
                try :
                    if len(spans) > 1:
                        director = spans[0].get_text().split("De\n")[1]
                        director = re.sub(r'\s+', ' ', director).strip()
                        actors = spans[2].get_text().split("Avec\n")[1]
                        actors = re.sub(r'\s+', ' ', actors).strip()
                    else : 
                        director = spans[0].get_text().split("De\n")[1]
                        director = re.sub(r'\s+', ' ', director).strip()
                        actors = ""
                except IndexError:
                    director = ""
                    actors = ""
                        
                results.append({
                    'week': week,
                    'title': title,
                    'director': director,
                    'actor': actors
                })
                
    month_movies = pd.DataFrame(results)

    return month_movies


def get_similar_movies(driver, row: dict[str, str], is_global: bool = False) -> pd.DataFrame:
    # Nettoyage simplifié de la query (on garde l'essentiel)
    query_clean = row["title"].strip()
    
    # Encodage propre de l'URL (gère les espaces, accents, caractères spéciaux)
    # Cela remplace avantageusement votre 'char_dict'
    query_encoded = quote(query_clean)
    url_main = f"https://www.imdb.com/find/?s=tt&q={query_encoded}&ref_=nv_sr_sm"
    
    driver.get(url_main)
    # Temps d'attente pour le rendu JS
    time.sleep(random.uniform(3, 5))
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    # Sélecteur plus précis pour les lignes de résultats
    items = soup.select('li.ipc-metadata-list-summary-item')[:6]
    
    results = []
    
    for li_tag in items:
        # 1. Extraction du Titre et de l'ID IMDb
        # On cherche le lien qui contient le titre
        link_tag = li_tag.select('a')[-1]
        if not link_tag:
            continue
            
        title = link_tag.get_text(strip=True)
        href = link_tag.get('href', '')
        
        # Extraction ID (ex: tt1234567)
        match_id = re.search(r'tt\d+', href)
        imdb_id = match_id.group() if match_id else ""

        # 2. Chercher acteurs pour vérifier la correspondance
        actor_search = str(li_tag.select('div.ipc-media')[0])
        act_match = round(sum([1 for actor in row["actor"].split(", ") if actor in actor_search]) / len(row["actor"].split(", ")) * 100,2)

        # 3. Extraction de l'année
        year_info = li_tag.select('div.ipc-media')
        year = 1492
        for span in year_info:
            match = re.search(r'\d{4}', str(span))
            if match:
                year = int(match.group())
                break
        
        # 3. Calcul de similarité
        similarity = SequenceMatcher(None, query_clean.lower(), title.lower()).ratio()
        
        # Filtrage selon vos critères (Année > 2022 et similitude > 0.6)
        if similarity >= 0.6 and (not is_global or year > 2022):
            results.append({
                "imdb_id": imdb_id,
                "extract_title": title,
                "act_match": act_match,
                "query": query_clean,
                "year": year,
                "similarity": round(similarity, 3)
            })

    # Gestion du cas vide
    if not results:
        results = [{"imdb_id": "", "extract_title": "", "act_match": "", "query": query_clean, "year": "", "similarity": ""}]

    final_df = pd.DataFrame(results)
    if not is_global:
        final_df = final_df.drop(["act_match"], axis=1)

    return final_df


In [5]:
NEW_ORDER = ["week", "imdb_id", "query", "director", "actor", "act_match", "year", "similarity", "extract_title", "title"]

In [10]:
month_movies = get_month_movies(int(2026), "november")
chrome_version = get_chrome_version_windows()

In [22]:
driver = init_driver(chrome_version)

In [23]:
try:
    total_results = []
    total_rows = len(month_movies)

    for index, row in month_movies.iterrows():
        df_res = get_similar_movies(driver, row.to_dict(), is_global=True)
        if not df_res.empty:
            total_results.append(df_res)

    if total_results:
        df_final = pd.concat(total_results, ignore_index=True)
        new_df = pd.merge(month_movies, df_final, left_on="title", right_on="query", how="left")
        new_df = new_df[NEW_ORDER]
        new_df.to_excel("new_movies.xlsx", index=False)
finally:
    driver.quit()  # Fermeture garantie du navigateur